<a href="https://colab.research.google.com/github/kinhalepuja31-cmd/Online_Retail_II/blob/main/Task_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
df=pd.read_excel("/content/online_retail_II.xlsx",sheet_name="Year 2009-2010")

In [6]:
df2=pd.read_excel("/content/online_retail_II.xlsx",sheet_name="Year 2010-2011")

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [8]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[ns]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [12]:
print("Combine two dataset")
df=pd.concat([df,df2],ignore_index=True)
print(f"Total initial rows combined: {df.shape[0]}")

Combine two dataset
Total initial rows combined: 1067371


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [13]:
print("cleaning data")

cleaning data


In [15]:
df.head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [23]:
missing_count = df['Customer ID'].isnull().sum()
print(missing_count)

243007


In [26]:
missing_percentage =(missing_count/len(df))*100
print(missing_percentage)

22.766872999172733


Customer ID is essential for tracking repeat buying behavior, so we should drop these null rows from the dataset for this specific analysis.

In [27]:
df_cleaned = df.dropna(subset=['Customer ID'])

In [31]:
missing_count = df_cleaned['Customer ID'].isnull().sum()
print(missing_count)

0


In [29]:
print(len(df))

1067371


In [30]:
print(len(df_cleaned))

824364


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


now with valid customers, the next step is to filter out cancelled orders and negative quantities from df_cleaned, then calculate Repeat Purchase Rate.

In [47]:

df_cleaned.columns = df_cleaned.columns.str.strip()


df_analysis = df_cleaned.copy()

df_analysis['Invoice'] = df_analysis['Invoice'].astype(str)
df_valid = df_analysis[~df_analysis['Invoice'].str.startswith(('C', 'c'), na=False)]


df_valid = df_valid[(df_valid['Quantity'] > 0) & (df_valid['Price'] > 0)]
print(f" Final clean dataset size: {len(df_valid):,} rows")




 Final clean dataset size: 805,549 rows


In [48]:
# 5. Calculate Repeat Purchase Rate
customer_orders = df_valid.groupby('Customer ID')['Invoice'].nunique().reset_index()
customer_orders.columns = ['Customer ID', 'total_orders']

total_customers = len(customer_orders)
repeat_customers = len(customer_orders[customer_orders['total_orders'] > 1])
repeat_rate = (repeat_customers / total_customers) * 100

print(f"\n📊 Repeat Purchase Rate Results:")
print(f"• Total Unique Customers: {total_customers:,}")
print(f"• Repeat Customers (2+ orders): {repeat_customers:,}")
print(f"• Repeat Purchase Rate: {repeat_rate:.2f}%")


📊 Repeat Purchase Rate Results:
• Total Unique Customers: 5,878
• Repeat Customers (2+ orders): 4,255
• Repeat Purchase Rate: 72.39%


Segment Table

In [49]:
# 1. Group by Customer ID and count unique orders (Invoices)
customer_orders = df_valid.groupby('Customer ID')['Invoice'].nunique().reset_index()
customer_orders.columns = ['Customer ID', 'total_orders']

# 2. Define the rule to segment customers based on order count
def assign_segment(orders):
    if orders == 1:
        return 'One-time Buyer'
    elif 2 <= orders <= 4:
        return 'Regular Repeat Buyer'
    else:
        return 'Highly Loyal/VIP'

# 3. Apply the segmentation function
customer_orders['segment'] = customer_orders['total_orders'].apply(assign_segment)

# 4. Create the final summary table with counts and percentages
total_customers = len(customer_orders)
segment_table = customer_orders['segment'].value_counts().reset_index()
segment_table.columns = ['Customer Segment', 'Customer Count']
segment_table['Percentage (%)'] = (segment_table['Customer Count'] / total_customers) * 100

print("👑 CUSTOMER LOYALTY SEGMENTS:")
print(segment_table.to_string(index=False, formatters={'Customer Count': '{:,}'.format, 'Percentage (%)': '{:.2f}%'.format}))


👑 CUSTOMER LOYALTY SEGMENTS:
    Customer Segment Customer Count Percentage (%)
    Highly Loyal/VIP          2,161         36.76%
Regular Repeat Buyer          2,094         35.62%
      One-time Buyer          1,623         27.61%


In [53]:
# 1. Calculate order_amount explicitly to fix the KeyError

df_valid['order_amount'] = df_valid['Quantity'] * df_valid['Price']

# 2. Get the total financial spend per individual invoice/order

order_revenue = df_valid.groupby(['Customer ID', 'Invoice'])['order_amount'].sum().reset_index()

# 3. Sort chronologically by Invoice to find the order sequence for each customer

order_revenue = order_revenue.sort_values(by=['Customer ID', 'Invoice'])
order_revenue['order_rank'] = order_revenue.groupby('Customer ID').cumcount() + 1

# 4. Label orders as 'First-time' (rank == 1) or 'Repeat' (rank > 1)

order_revenue['customer_type'] = order_revenue['order_rank'].apply(lambda x: 'First-time' if x == 1 else 'Repeat')

# 5. Calculate the Average Order Value (AOV) for each group
aov_comparison = order_revenue.groupby('customer_type')['order_amount'].mean().reset_index()
aov_comparison.columns = ['Customer Type', 'Average Order Value (AOV)']

print("💰 AVERAGE ORDER VALUE (AOV) COMPARISON:")
print(aov_comparison.to_string(index=False, formatters={'Average Order Value (AOV)': '${:,.2f}'.format}))


💰 AVERAGE ORDER VALUE (AOV) COMPARISON:
Customer Type Average Order Value (AOV)
   First-time                   $416.41
       Repeat                   $491.97


In [54]:
# 1. Calculate order_amount explicitly to fix the KeyError

df_valid['order_amount'] = df_valid['Quantity'] * df_valid['Price']

# 2. Get the total financial spend per individual invoice/order

order_revenue = df_valid.groupby(['Customer ID', 'Invoice'])['order_amount'].sum().reset_index()

# 3. Sort chronologically by Invoice to find the order sequence for each customer

order_revenue = order_revenue.sort_values(by=['Customer ID', 'Invoice'])
order_revenue['order_rank'] = order_revenue.groupby('Customer ID').cumcount() + 1

# 4. Label orders as 'First-time' (rank == 1) or 'Repeat' (rank > 1)

order_revenue['customer_type'] = order_revenue['order_rank'].apply(lambda x: 'First-time' if x == 1 else 'Repeat')

# 5. Calculate the Average Order Value (AOV) for each group

aov_comparison = order_revenue.groupby('customer_type')['order_amount'].mean().reset_index()
aov_comparison.columns = ['Customer Type', 'Average Order Value (AOV)']

print("💰 AVERAGE ORDER VALUE (AOV) COMPARISON:")
print(aov_comparison.to_string(index=False, formatters={'Average Order Value (AOV)': '${:,.2f}'.format}))


💰 AVERAGE ORDER VALUE (AOV) COMPARISON:
Customer Type Average Order Value (AOV)
   First-time                   $416.41
       Repeat                   $491.97
